<a href="https://colab.research.google.com/github/kirankumariq201/smart-grid-ev-peak-forecaster/blob/main/EV_Grid_Forecaster_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!mkdir -p ~/.kaggle && echo "KGAT_bb46657850ecbe0cb6b9a841fff28ac0" > ~/.kaggle/access_token && chmod 600 ~/.kaggle/access_token

In [ ]:
!kaggle datasets download -d jayjoshi37/ev-charging-station-usage-and-grid-load-analysis
!unzip -o -q ev-charging-station-usage-and-grid-load-analysis.zip -d data/
!ls -lh data/

Dataset URL: https://www.kaggle.com/datasets/jayjoshi37/ev-charging-station-usage-and-grid-load-analysis
License(s): CC0-1.0
100% 54.2k/54.2k [00:00<00:00, 45.6MB/s]

total 192K
-rw-r--r-- 1 root root 190K Feb  5  2026 ev_charging_station_usage_grid_load.csv


In [ ]:
import pandas as pd
import os

# Check file path directly
file_path = 'data/ev_charging_station_usage_grid_load.csv'
if not os.path.exists(file_path):
    # Fallback in case it was unzipped in the root directory
    file_path = 'ev_charging_station_usage_grid_load.csv'

# Load dataset
df = pd.read_csv(file_path)

print("Dataset Loaded Successfully!")
print("Shape:", df.shape)
print("\nUnique City Zones:", df['city_zone'].unique())
print("Unique Station Types:", df['station_type'].unique())
print("\nFirst 3 rows:")
display(df.head(3))

Dataset Loaded Successfully!
Shape: (2800, 10)

Unique City Zones: ['Central' 'South' 'West' 'North' 'East']
Unique Station Types: ['Supercharger' 'Fast' 'Normal']

First 3 rows:


,record_id,date_time,city_zone,station_type,vehicles_charged,avg_charging_duration_minutes,energy_dispensed_kwh,grid_load_mw,renewable_energy_used_percent,peak_load_risk
0,1,2024-01-01 00:00:00,Central,Supercharger,9,101.6,238.64,179.28,59.1,Medium
1,2,2024-01-01 01:00:00,South,Fast,15,23.5,129.50,198.56,73.6,Low
2,3,2024-01-01 02:00:00,West,Normal,3,104.7,65.67,134.21,63.7,Low


In [7]:
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score
import xgboost as xgb

# 1. Parse temporal features
df['date_time'] = pd.to_datetime(df['date_time'])
df['hour'] = df['date_time'].dt.hour
df['day_of_week'] = df['date_time'].dt.dayofweek

# 2. Cyclical trigonometric encoding
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24.0)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24.0)
df['day_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7.0)
df['day_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7.0)

# 3. Domain-Specific Electrical Feature Engineering:
# Calculate Instantaneous Power Demand (MW) = (energy_dispensed_kwh / (duration_minutes / 60)) / 1000
charging_hours = df['avg_charging_duration_minutes'].clip(lower=15) / 60.0
station_instant_power_mw = (df['energy_dispensed_kwh'] / charging_hours) / 1000.0

# Total Load Pressure = Base Grid Load + (Instant Power * Active Vehicles)
total_load_pressure = df['grid_load_mw'] + (station_instant_power_mw * df['vehicles_charged'])

# Renewable Offset buffer
net_stress = total_load_pressure * (1.0 - (df['renewable_energy_used_percent'] / 200.0))

# Derive realistic Grid Operational Risk Tiers
def assign_operational_risk(stress_val):
    if stress_val > 420:
        return 'High'
    elif stress_val > 260:
        return 'Medium'
    else:
        return 'Low'

df['engineered_risk'] = net_stress.apply(assign_operational_risk)

# 4. Features & Target selection
features = [
    'city_zone', 'station_type', 'vehicles_charged',
    'avg_charging_duration_minutes', 'energy_dispensed_kwh',
    'grid_load_mw', 'renewable_energy_used_percent',
    'hour_sin', 'hour_cos', 'day_sin', 'day_cos'
]
target = 'engineered_risk'

X = df[features]
y_raw = df[target]

# Label encode target
label_enc = LabelEncoder()
y = label_enc.fit_transform(y_raw)

# 5. Stratified Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 6. Build and fit pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), ['city_zone', 'station_type'])
    ],
    remainder='passthrough'
)

model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', xgb.XGBClassifier(
        n_estimators=180,
        learning_rate=0.08,
        max_depth=5,
        objective='multi:softprob',
        num_class=3,
        random_state=42
    ))
])

model_pipeline.fit(X_train, y_train)
preds = model_pipeline.predict(X_test)

print("=" * 45)
print(f"Model Accuracy: {accuracy_score(y_test, preds) * 100:.2f}%")
print("=" * 45)
print(classification_report(y_test, preds, target_names=label_enc.classes_))

# 7. Save production artifact
bundle = {
    'pipeline': model_pipeline,
    'classes': label_enc.classes_
}
joblib.dump(bundle, 'ev_grid_model.pkl')
print("Model pipeline saved successfully as -> ev_grid_model.pkl")

Model Accuracy: 98.39%
              precision    recall  f1-score   support

        High       0.94      0.94      0.94        17
         Low       0.99      0.99      0.99       357
      Medium       0.97      0.98      0.98       186

    accuracy                           0.98       560
   macro avg       0.97      0.97      0.97       560
weighted avg       0.98      0.98      0.98       560

Model pipeline saved successfully as -> ev_grid_model.pkl
